In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns

class FeatureImportanceAnalyzer:
    """?? ??? ?? ? ?? ?? ???"""
    
    def __init__(self):
        self.feature_importances = {}
        self.selected_features = {}
        self.submissions = {}
        
    def load_and_preprocess(self, random_seed=42):
        """?? ???"""
        np.random.seed(random_seed)
        
        train_df = pd.read_csv('train.csv')
        test_df = pd.read_csv('test.csv')
        
        feature_cols = [col for col in train_df.columns if col not in ['ID', 'Cancer']]
        
        X_train = train_df[feature_cols].copy()
        y_train = train_df['Cancer'].copy()
        X_test = test_df[feature_cols].copy()
        
        print(f"?? ?? ?? ??: {len(feature_cols)}")
        print(f"   ?? ??: {feature_cols}")
        
        # ????? ???
        categorical_cols = X_train.select_dtypes(include=['object']).columns
        label_encoders = {}
        
        for col in categorical_cols:
            le = LabelEncoder()
            X_train[col] = le.fit_transform(X_train[col].astype(str))
            label_encoders[col] = le
            
            test_values = X_test[col].astype(str)
            test_encoded = []
            for val in test_values:
                if val in le.classes_:
                    test_encoded.append(le.transform([val])[0])
                else:
                    test_encoded.append(0)
            X_test[col] = test_encoded
        
        # ??? ??
        numeric_cols = X_train.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            median_val = X_train[col].median()
            X_train[col].fillna(median_val, inplace=True)
            X_test[col].fillna(median_val, inplace=True)
        
        return X_train, y_train, X_test, test_df['ID'], feature_cols
    
    def analyze_feature_importance(self, X_train, y_train, feature_cols):
        """??? ???? ?? ??? ??"""
        print("\n?? ?? ??? ?? ?...")
        
        pos_count = (y_train == 1).sum()
        neg_count = (y_train == 0).sum()
        scale_pos_weight = neg_count / pos_count
        
        # 1. XGBoost ???
        print("  XGBoost ??? ??...")
        xgb_model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            scale_pos_weight=scale_pos_weight
        )
        xgb_model.fit(X_train, y_train)
        xgb_importance = xgb_model.feature_importances_
        
        # 2. LightGBM ???  
        print("  LightGBM ??? ??...")
        lgb_model = lgb.LGBMClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            class_weight='balanced',
            verbose=-1
        )
        lgb_model.fit(X_train, y_train)
        lgb_importance = lgb_model.feature_importances_
        
        # 3. Random Forest ???
        print("  Random Forest ??? ??...")
        rf_model = RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            random_state=42,
            class_weight='balanced'
        )
        rf_model.fit(X_train, y_train)
        rf_importance = rf_model.feature_importances_
        
        # 4. ??? ?? (F-test)
        print("  F-test ??? ??...")
        f_selector = SelectKBest(f_classif, k='all')
        f_selector.fit(X_train, y_train)
        f_scores = f_selector.scores_
        f_importance = f_scores / f_scores.max()  # ???
        
        # 5. Mutual Information
        print("  Mutual Information ??...")
        mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
        mi_importance = mi_scores / mi_scores.max()  # ???
        
        # ?? ??? ??
        self.feature_importances = {
            'XGBoost': xgb_importance,
            'LightGBM': lgb_importance,
            'RandomForest': rf_importance,
            'F_test': f_importance,
            'MutualInfo': mi_importance
        }
        
        # ??? ?????? ??
        importance_df = pd.DataFrame({
            'Feature': feature_cols,
            'XGBoost': xgb_importance,
            'LightGBM': lgb_importance,
            'RandomForest': rf_importance,
            'F_test': f_importance,
            'MutualInfo': mi_importance
        })
        
        # ?? ??? ??
        importance_df['Average'] = importance_df[['XGBoost', 'LightGBM', 'RandomForest', 'F_test', 'MutualInfo']].mean(axis=1)
        importance_df = importance_df.sort_values('Average', ascending=False)
        
        print("\n?? ?? ??? ??:")
        for i, row in importance_df.iterrows():
            print(f"  {row['Feature']:20s}: {row['Average']:.4f} (XGB: {row['XGBoost']:.3f}, LGB: {row['LightGBM']:.3f}, RF: {row['RandomForest']:.3f})")
        
        return importance_df
    
    def test_feature_combinations(self, X_train, y_train, X_test, test_ids, importance_df):
        """??? ?? ?? ???"""
        print("\n?? ??? ?? ?? ???...")
        
        pos_count = (y_train == 1).sum()
        neg_count = (y_train == 0).sum()
        scale_pos_weight = neg_count / pos_count
        
        # ???? ?? ???
        feature_counts = [5, 7, 8, 10, 12]
        
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        
        best_score = 0
        best_features = None
        best_count = 0
        best_predictions = None
        
        for k in feature_counts:
            print(f"\n  ?? {k}? ???? ???...")
            
            # ?? k? ?? ??
            top_features = importance_df.head(k)['Feature'].tolist()
            print(f"    ??? ??: {top_features}")
            
            X_train_selected = X_train[top_features]
            X_test_selected = X_test[top_features]
            
            # XGBoost? ?? ???
            model = xgb.XGBClassifier(
                n_estimators=150,
                max_depth=6,
                learning_rate=0.08,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                scale_pos_weight=scale_pos_weight,
                reg_alpha=0.1,
                reg_lambda=0.1
            )
            
            # Cross Validation
            cv_scores = cross_val_score(model, X_train_selected, y_train, cv=cv, scoring='f1')
            cv_mean = cv_scores.mean()
            cv_std = cv_scores.std()
            
            print(f"    CV F1: {cv_mean:.6f} ? {cv_std:.6f}")
            
            if cv_mean > best_score:
                best_score = cv_mean
                best_features = top_features
                best_count = k
                
                # ?? ?? ??? ??
                model.fit(X_train_selected, y_train)
                best_predictions = model.predict(X_test_selected)
        
        print(f"\n? ?? ??: {best_count}? ?? (CV F1: {best_score:.6f})")
        print(f"   ??? ??: {best_features}")
        
        submission = pd.DataFrame({'ID': test_ids, 'Cancer': best_predictions})
        self.submissions['feature_selected'] = submission
        self.selected_features['best'] = best_features
        
        return submission, best_features
    
    def ensemble_with_selected_features(self, X_train, y_train, X_test, test_ids, selected_features):
        """??? ???? ???"""
        print(f"\n?? ??? {len(selected_features)}? ???? ???...")
        
        X_train_selected = X_train[selected_features]
        X_test_selected = X_test[selected_features]
        
        pos_count = (y_train == 1).sum()
        neg_count = (y_train == 0).sum()
        scale_pos_weight = neg_count / pos_count
        
        # ??? ??? ???
        models = {
            'xgb_conservative': xgb.XGBClassifier(n_estimators=180, max_depth=5, learning_rate=0.07, random_state=42, scale_pos_weight=scale_pos_weight),
            'xgb_balanced': xgb.XGBClassifier(n_estimators=160, max_depth=6, learning_rate=0.08, random_state=123, scale_pos_weight=scale_pos_weight),
            'lgb': lgb.LGBMClassifier(n_estimators=160, max_depth=6, learning_rate=0.08, random_state=42, class_weight='balanced', verbose=-1),
            'rf': RandomForestClassifier(n_estimators=160, max_depth=8, random_state=42, class_weight='balanced')
        }
        
        # Cross-validation?? ? ?? ?? ??
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        model_scores = {}
        model_predictions = {}
        
        for name, model in models.items():
            cv_scores = cross_val_score(model, X_train_selected, y_train, cv=cv, scoring='f1')
            model_scores[name] = cv_scores.mean()
            print(f"  {name}: CV F1 = {cv_scores.mean():.6f}")
            
            # ?? ???? ??
            model.fit(X_train_selected, y_train)
            model_predictions[name] = model.predict_proba(X_test_selected)[:, 1]
        
        # ?? ?? ?? ???
        total_score = sum(model_scores.values())
        weights = [score/total_score for score in model_scores.values()]
        
        print(f"  ???: {dict(zip(model_scores.keys(), weights))}")
        
        # ?? ??
        ensemble_proba = np.average(list(model_predictions.values()), axis=0, weights=weights)
        ensemble_predictions = (ensemble_proba > 0.5).astype(int)
        
        submission = pd.DataFrame({'ID': test_ids, 'Cancer': ensemble_predictions})
        self.submissions['feature_ensemble'] = submission
        
        return submission
    
    def ultra_simple_model(self, X_train, y_train, X_test, test_ids, selected_features):
        """????? ??? ??"""
        print(f"\n?? ??? ???: ?? 5? ??? ??")
        
        # ?? 5?? ??
        top5_features = selected_features[:5]
        print(f"   ?? ??: {top5_features}")
        
        X_train_simple = X_train[top5_features]
        X_test_simple = X_test[top5_features]
        
        pos_count = (y_train == 1).sum()
        neg_count = (y_train == 0).sum()
        scale_pos_weight = neg_count / pos_count
        
        # ?? ??? XGBoost
        simple_model = xgb.XGBClassifier(
            n_estimators=100,  # ?? ??
            max_depth=4,       # ?? ??
            learning_rate=0.1, 
            random_state=42,
            scale_pos_weight=scale_pos_weight
        )
        
        # CV ??
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(simple_model, X_train_simple, y_train, cv=cv, scoring='f1')
        
        print(f"   CV F1: {cv_scores.mean():.6f} ? {cv_scores.std():.6f}")
        
        # ?? ? ??
        simple_model.fit(X_train_simple, y_train)
        simple_predictions = simple_model.predict(X_test_simple)
        
        submission = pd.DataFrame({'ID': test_ids, 'Cancer': simple_predictions})
        self.submissions['ultra_simple'] = submission
        
        return submission

def run_feature_selection_optimization():
    """?? ?? ?? ??? ??"""
    print("?? ?? ??? ?? ??? ??!")
    print("??? ??? ??? ?? ??? ? ????!")
    print("=" * 60)
    
    analyzer = FeatureImportanceAnalyzer()
    
    # 1. ??? ?? ? ???
    X_train, y_train, X_test, test_ids, feature_cols = analyzer.load_and_preprocess()
    
    # 2. ?? ??? ??
    importance_df = analyzer.analyze_feature_importance(X_train, y_train, feature_cols)
    
    # 3. ??? ?? ?? ???
    best_submission, best_features = analyzer.test_feature_combinations(
        X_train, y_train, X_test, test_ids, importance_df
    )
    
    # 4. ??? ???? ???
    ensemble_submission = analyzer.ensemble_with_selected_features(
        X_train, y_train, X_test, test_ids, best_features
    )
    
    # 5. ??? ???
    simple_submission = analyzer.ultra_simple_model(
        X_train, y_train, X_test, test_ids, best_features
    )
    
    # 6. ?? ??
    submissions_to_save = [
        ('feature_selected_best.csv', best_submission, "?? ?? ??"),
        ('feature_ensemble.csv', ensemble_submission, "??? ?? ???"),
        ('ultra_simple.csv', simple_submission, "??? ???")
    ]
    
    print(f"\n?? ?? ?? ??...")
    for filename, submission, description in submissions_to_save:
        submission.to_csv(filename, index=False)
        
        # ?? ?? ??
        pred_dist = submission['Cancer'].value_counts()
        print(f"  {filename}: {description}")
        print(f"    ?? ?? - 0: {pred_dist.get(0, 0)}, 1: {pred_dist.get(1, 0)}")
    
    print(f"\n?? ?? ????:")
    print(f"1. feature_ensemble.csv ? (??? ?? ???)")
    print(f"2. feature_selected_best.csv (?? ?? ??)")
    print(f"3. ultra_simple.csv (??? ???)")
    
    print(f"\n?? ?? ????:")
    print(f"- ???? ?? ???? ???? ?? ? ??")
    print(f"- ??? ??? ??? ??? ? ?? ? ??")
    print(f"- ???? ??? ?? ?? ?? ??")
    
    # ?? ??? ??? (????)
    print(f"\n?? ?? ??? ?? ??!")
    print(f"?? ???: {best_features}")

if __name__ == "__main__":
    run_feature_selection_optimization()